In [ ]:
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt

# Define um estilo visual mais agradável para os gráficos
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported and style set.")

In [ ]:
# Célula 2 (Com Ordenação Natural)

import pandas as pd
import os
import glob

results_dir = 'results'
experiment_data = []

stat_files = glob.glob(os.path.join(results_dir, '**', '*_stats.txt'), recursive=True)

for file_path in stat_files:
    data = {}
    with open(file_path, 'r') as f:
        next(f, None) # Pula a primeira linha "--- ..."
        for line in f:
            if ':' in line:
                key, value = line.split(':', 1)
                data[key.strip()] = value.strip()
    
    instance_info = os.path.basename(os.path.dirname(file_path))
    data['instance_base'] = instance_info.split('_lambda_')[0]
    
    experiment_data.append(data)

df_results = pd.DataFrame(experiment_data)

# Converte as colunas para o tipo numérico usando as chaves corretas
df_results['objective_value'] = pd.to_numeric(df_results['Objective_Value'])
df_results['num_clusters'] = pd.to_numeric(df_results['Number_of_Clusters'])
df_results['execution_time_s'] = pd.to_numeric(df_results['Solver_Execution_Time_s'])
df_results['lambda'] = pd.to_numeric(df_results['Lambda Weight'])

# --- CORREÇÃO DA ORDENAÇÃO ---
# 1. Extrai o número da instância (ex: 1, 2, ..., 10) para uma nova coluna
df_results['instance_num'] = df_results['instance_base'].str.extract(r'P(\d+)').astype(int)

# 2. Ordena primeiro pelo número da instância, depois pelo lambda
df_results.sort_values(by=['instance_num', 'lambda'], inplace=True)

# 3. (Opcional) Remove a coluna temporária após a ordenação
df_results.drop(columns=['instance_num'], inplace=True)

print(f"Loaded and processed {len(df_results)} experiment results successfully.")

# Agora o .head() mostrará os resultados do P1, como esperado
display(df_results[['instance_base', 'lambda', 'objective_value', 'num_clusters', 'execution_time_s']])

In [ ]:
# Pega uma lista ordenada de todas as instâncias únicas
unique_instances = sorted(df_results['instance_base'].unique())

# Itera sobre cada nome de instância na lista
for instance_name in unique_instances:
    df_instance = df_results[df_results['instance_base'] == instance_name].copy()

    print("="*80)
    print(f"ANALYSIS FOR INSTANCE: {instance_name}")
    print("="*80)

    # --- 1. Plot da Fronteira de Pareto ---
    fig, ax = plt.subplots(figsize=(12, 8))
    
    ax.plot(
        df_instance['num_clusters'], 
        df_instance['objective_value'], 
        marker='o', 
        markersize=10,
        linestyle='--',
        color='b',
        label='Soluções de Trade-off'
    )

    for i, row in df_instance.iterrows():
        ax.text(
            row['num_clusters'] + 0.1, 
            row['objective_value'], 
            f" λ={row['lambda']}", 
            fontsize=12,
            verticalalignment='bottom'
        )

    ax.set_xlabel('Número de Clusters (k) - (Simplicidade)', fontsize=14)
    ax.set_ylabel('Desequilíbrio Esperado - (Qualidade)', fontsize=14)
    ax.set_title(f'Fronteira de Pareto para a Instância: {instance_name}', fontsize=16, pad=20)
    ax.legend(fontsize=12)
    ax.grid(True)
    
    output_fig_path = os.path.join('results', f'pareto_{instance_name}.png')
    plt.savefig(output_fig_path, dpi=300)
    print(f"Pareto front plot saved to '{output_fig_path}'")
    plt.show()

    # --- 2. Análise das Soluções de Destaque ---
    print(f"\n--- Soluções de Destaque para: {instance_name} ---")
    
    # Solução com Foco em SIMPLICIDADE (lambda=0.0)
    solution_simple = df_instance[df_instance['lambda'] == 0.0]
    print("\n[+] Solução Focada em SIMPLICIDADE (lambda=0.0):")
    if not solution_simple.empty:
        display(solution_simple[['num_clusters', 'objective_value']])
    
    # Solução de COMPROMISSO (lambda=0.5)
    solution_balanced = df_instance[df_instance['lambda'] == 0.5]
    print("\n[+] Solução de COMPROMISSO (lambda=0.5):")
    if not solution_balanced.empty:
        display(solution_balanced[['num_clusters', 'objective_value']])

    # Solução com Foco em QUALIDADE (lambda=1.0)
    solution_quality = df_instance[df_instance['lambda'] == 1.0]
    print("\n[+] Solução Focada em QUALIDADE (lambda=1.0):")
    if not solution_quality.empty:
        display(solution_quality[['num_clusters', 'objective_value']])
    
    print("\n\n")